In [ ]:
import numpy as np

from plotly import graph_objects as go

In [ ]:
Zeitreihen = []

In [ ]:
sin, cos, sin 2 freq, cos 2 freq, changepoint sin -> sin(2.), changepoint sin(2.) -> sin

In [ ]:
resolution = 800

In [ ]:
reihe = np.zeros(resolution)
for t in range(resolution):
    reihe[t] = np.sin(2*t*np.pi/(resolution/10))
    
Zeitreihen.append(reihe)

In [ ]:
reihe = np.zeros(resolution)
for t in range(resolution):
    reihe[t] = np.cos(2*t*np.pi/(resolution/10))
    
Zeitreihen.append(reihe)

In [ ]:
reihe = np.zeros(resolution)
for t in range(resolution):
    reihe[t] = np.sin(2*t*np.pi/(3*resolution/20))
    
Zeitreihen.append(reihe)

In [ ]:
reihe = np.zeros(resolution)
for t in range(resolution):
    reihe[t] = np.cos(2*t*np.pi/(3*resolution/20))
    
Zeitreihen.append(reihe)

In [ ]:
reihe = np.zeros(resolution)
for t in range(resolution//2):
    reihe[t] = np.cos(2*t*np.pi/20)
    reihe[resolution//2+t] = np.cos(2*t*np.pi/40)
    
Zeitreihen.append(reihe)

In [ ]:
reihe = np.zeros(resolution)
for t in range(resolution//2):
    reihe[resolution//2+t] = np.cos(2*t*np.pi/20)
    reihe[t] = np.cos(2*t*np.pi/40)
    
Zeitreihen.append(reihe)

In [ ]:
go.Figure(go.Scatter(x=list(range(800)), y=Zeitreihen[5]))

In [ ]:
for pos in range(len(Zeitreihen)):
    reihe = Zeitreihen[pos]
    
    with open("/Users/rolf.bardeli1/Documents/docs/math/TDA_cha4_sprint_rescue/src/timeseries_%d.csv"%(pos), "wt") as ofile:
        for t in range(resolution):
            ofile.write("%d,%f\n"%(t, reihe[t]))

In [ ]:
from gtda.time_series import SingleTakensEmbedding

embedding_dimension_periodic = 3
embedding_time_delay_periodic = 8
stride = 2

embedder_periodic = SingleTakensEmbedding(
    parameters_type="fixed",
    n_jobs=2,
    time_delay=embedding_time_delay_periodic,
    dimension=embedding_dimension_periodic,
    stride=stride,
)

In [ ]:
embeddings = []
for reihe in Zeitreihen:
    embeddings.append(embedder_periodic.fit_transform(reihe))

In [ ]:
from gtda.plotting import plot_point_cloud

plot_point_cloud(embeddings[4])

In [ ]:
pos = 0
for embedding in embeddings:
    with open("/Users/rolf.bardeli1/Documents/docs/math/TDA_cha4_sprint_rescue/src/embedding_%d.csv"%(pos), "wt") as ofile:
        for row in embedding:
            ofile.write("%f,%f,%f,1\n"%(row[0], row[1], row[2]))
            
    pos += 1

In [ ]:
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import BettiCurve, PersistenceLandscape
from gtda.plotting import plot_diagram

In [ ]:
VR = VietorisRipsPersistence()#homology_dimensions=(0, 1, 2))

In [ ]:
persistence = []
for embedding in embeddings:
    persistence.append(VR.fit_transform([embedding]))

In [ ]:
for num in range(len(persistence)):
    fig = plot_diagram(persistence[num][0])
    fig.update_xaxes(range=[0, 1.7])
    fig.update_yaxes(range=[0, 1.7])
    
    # remove axis titles
    fig.update_xaxes(title_text='')
    fig.update_yaxes(title_text='')
    
    fig.data[2].marker.color = 'blue'
    fig.data[2].marker.symbol = 'square'
    fig.write_image("/Users/rolf.bardeli1/Documents/docs/math/TDA_cha4_sprint_rescue/src/ts_persist_na_%d.pdf"
                       %(num))

In [ ]:
from gtda.diagrams import PersistenceLandscape

In [ ]:
landscape = PersistenceLandscape(n_bins=100)

In [ ]:
landscapes = [landscape.fit_transform(reihe) for reihe in persistence]

In [ ]:
dist = np.zeros((6,6))
for row in range(6):
    for column in range(6):
        dist[row, column] = np.linalg.norm(landscapes[row][0][0] - landscapes[column][0][0])
        dist[row, column] += np.linalg.norm(landscapes[row][0][1] - landscapes[column][0][1])

In [ ]:
for row in (100 * dist / dist.max()).round():
    print(row)

In [ ]:
embeddings[5].shape

In [ ]:
slices = []
for t in range(0, 340, 5):
    slices.append(embeddings[5][t:t+50])

In [ ]:
slice_pers = VR.fit_transform(slices)

In [ ]:
len(slice_pers)

In [ ]:
fig = plot_diagram(slice_pers[50])
fig.update_xaxes(range=[0, 2.2])
fig.update_yaxes(range=[0, 2.2])

# remove axis titles
fig.update_xaxes(title_text='')
fig.update_yaxes(title_text='')

fig.data[2].marker.color = 'blue'
fig.data[2].marker.symbol = 'square'
fig.write_image("/Users/rolf.bardeli1/Documents/docs/math/TDA_cha4_sprint_rescue/src/ts_slice_persist_na_50.pdf")

In [ ]:
slice_landscapes = landscape.fit_transform(slice_pers)

In [ ]:
slice_landscapes[0][0].shape

In [ ]:
distances = []
for slicepos in range(len(slice_landscapes)-1):
    dist = np.linalg.norm(slice_landscapes[slicepos][0] - slice_landscapes[slicepos+1][0])
    dist += np.linalg.norm(slice_landscapes[slicepos][1] - slice_landscapes[slicepos+1][1])
    distances.append(dist)

In [ ]:
with open("/Users/rolf.bardeli1/Documents/docs/math/TDA_cha4_sprint_rescue/src/changepoint.csv", "wt") as ofile:
    for pos in range(len(distances)):
        ofile.write("%d,%f,1\n"%(pos,distances[pos]))